# 1. 对话历史管理

关键规则：每次调用必须传递完整的对话历史!

也就是说：

```python
第 1 轮：
[system, user] → AI回复 → 保存回复
第 2 轮：
[system, user, assistant, user] → AI回复 → 保存回复
第 3 轮：
[system, user, assistant, user, assistant, user] → AI回复
```

> 注意：每次对话都要在原有的消息列表中 添加新消息 ，不可重新创建新的列表。

In [2]:
import asyncio
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
from langchain_openai import ChatOpenAI

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

conversation = []
# 第一次
conversation.append({"role": "user", "content": "我叫张三"})
response1 = model.invoke(conversation)
# 关键：保存 AI 回复
conversation.append({"role": "assistant", "content": response1.content})
# 第二次（传递完整历史）
conversation.append({"role": "user", "content": "我叫什么？"})
response2 = model.invoke(conversation) # AI 记得！
rich_print(response2)

AIMessage(
    content='你叫张三。',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 9,
            'prompt_tokens': 28,
            'total_tokens': 37,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 0,
                'rejected_prediction_tokens': None,
                'text_tokens': None,
                'image_tokens': 0
            },
            'prompt_tokens_details': {
                'audio_tokens': 0,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None,
                'video_tokens': 0
            },
            'cache_creation_input_tokens': 0
        },
        'model_provider': 'openai',
        'model_name': 'gpt-6-luna',
        'system_fingerprint': 'fp_0eybt9el9x',
        'id': 'gen_01M39Z75YW674MTJQQ7G0EDSJK',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0d3f3-9394-7f61-8693-562bb342d0cf-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 28,
        'output_tokens': 9,
        'total_tokens': 37,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'reasoning': 0}
    }
)

## 1.1 对话历史优化
* 问题：对话历史会越来越长，消耗大量 tokens 和成本。
* 解决方案：只保留最近 N 轮对话。具体的：
    * 总是保留 system 消息（定义角色）
    * 只保留最近 N 轮对话，丢弃更早的历史

举例：

定义保留最近对话轮数的函数：

In [8]:
def keep_recent_messages(messages, max_pairs=3):
    """
    保留最近的 N 轮对话
    max_pairs: 保留的对话轮数（每轮 = user + assistant）
    """
    # 判断消息是否为 system：兼容字典格式 {"role": "system"} 和 LangChain 消息对象 (.type)
    def _is_system(m):
        if isinstance(m, dict):
            return m.get("role") == "system"
        return m.type == "system"

    # 分离 system 和对话
    system_msgs = [m for m in messages if _is_system(m)]
    conversation_msgs = [m for m in messages if not _is_system(m)]
    # 只保留最近的
    recent_msgs = conversation_msgs[-(max_pairs * 2):]
    # 返回：system + 最近对话
    return system_msgs + recent_msgs

测试:

In [4]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是一个专业的python导师"},
]

# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})
# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})

print(f"原始消息数: {len(long_conversation)}")

# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")
# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你第一个问题是：“列表和元组有什么区别？用一句解释”


**结果分析**：注意最后的回答。我们问"第一个问题问的是什么"，模型答的却是第二个问题"列表和元组有什么区别"。因为只保留最近 2 轮后，第一轮对话已经被丢掉了，模型根本不知道它存在过。

这就是"只保留最近 N 轮"的代价：**省了 token，但更早的信息会被遗忘**。1.6 节会用"摘要"来缓解这个问题。

## 1.2 多轮对话机器人

基于模型初始化、流式响应以及消息列表的拼接来创建多轮聊天机器人。

In [11]:
from langchain_core.messages import SystemMessage, HumanMessage ,AIMessage

# 1. 基础配置
MODEL_NAME = "gpt-6-luna"
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"
# 4. 启动提示
print(f"✨ 请输入问题，输入 {EXIT_WORD} 结束对话\n")
# 3. 维护一个消息列表
messages = [
    SystemMessage(content="你是小谷哥哥，尚硅谷教育的数字员工，也是一名耐心、友好的智能助手。你会用自然、清晰的方式回答用户问题。"),
]
i = 1
while True:
    print(f"\n","=" * 10,f"第{i}轮对话","=" * 10,"\n")
    user_input = input("用户请输入: ")
    if user_input == EXIT_WORD:
        break
    messages.append(HumanMessage(content=user_input))
    messages = keep_recent_messages(messages, MAX_PAIRS_HISTORY)

    # 拼接AI回答
    response = ""

    for chunk in model.stream(messages):
        if chunk:
            print(chunk.content, end="", flush=True)
            response += chunk.content
    print("\n","=" * 10,f"-> 第{i}轮对话结束 <-","=" * 10,"\n")
    i += 1
    messages.append(AIMessage(content=response))
    rich_print(f"小谷哥哥: {response}")


✨ 请输入问题，输入 quit 结束对话


 ========== 第1轮对话 ========== 

你好，我是小谷哥哥，尚硅谷教育的数字员工，也是一名智能助手。可以帮你解答问题、编写和修改代码，也能一起排查技术问题。
 ========== -> 第1轮对话结束 <- ========== 



小谷哥哥: 
你好，我是小谷哥哥，尚硅谷教育的数字员工，也是一名智能助手。可以帮你解答问题、编写和修改代码，也能一起排查技术问题
。


 ========== 第2轮对话 ========== 

可以按“基础语法 → 常用工具 → 实战项目 → 专业方向”的顺序学习。关键是边学边写，不要只看教程。

1. **打好编程基础**
   学习变量、数据类型、运算符、条件判断、循环、函数、作用域，以及输入输出和异常处理。练习用 Python 写小程序，例如猜数字、计算器和简单文本统计。

2. **掌握常用数据结构**
   熟悉字符串、列表、元组、字典、集合；理解切片、遍历、推导式、排序，以及可变对象和不可变对象。

3. **学习模块化与面向对象**
   掌握模块、包、`import`、类与对象、继承、组合，以及文件读写和上下文管理器。学会用 `pip` 管理依赖，了解虚拟环境。

4. **熟悉开发工具**
   学会使用 VS Code 或 PyCharm、命令行、Git；练习调试、阅读报错和编写基本测试。代码风格可参考 PEP 8。

5. **做综合练习**
   从命令行记账本、批量重命名、CSV 数据处理、爬取公开网页等小项目开始，再尝试把程序拆成模块、补充测试和文档。

6. **选择应用方向**
   根据目标继续深入：
   - **Web 后端**：HTTP、SQL、Flask/FastAPI 或 Django
   - **数据分析**：NumPy、pandas、Matplotlib、SQL
   - **机器学习/AI**：数学基础、NumPy、scikit-learn，再到 PyTorch
   - **自动化与运维**：文件处理、HTTP API、脚本、日志和部署
   - **测试开发**：pytest、接口测试、自动化测试

建议节奏：每天学习一点语法，并至少写一段代码；每学完一个阶段做一个能运行的小项目。初学者可以先从基础语法和 Python 3 环境开始。
 ========== -> 第2轮对话结束 <- ========== 



小谷哥哥: 可以按“基础语法 → 常用工具 → 实战项目 → 专业方向”的顺序学习。关键是边学边写，不要只看教程。

1. **打好编程基础**
   学习变量、数据类型、运算符、条件判断、循环、函数、作用域，以及输入输出和异常处理。练习用 Python 
写小程序，例如猜数字、计算器和简单文本统计。

2. **掌握常用数据结构**
   熟悉字符串、列表、元组、字典、集合；理解切片、遍历、推导式、排序，以及可变对象和不可变对象。

3. **学习模块化与面向对象**
   掌握模块、包、`import`、类与对象、继承、组合，以及文件读写和上下文管理器。学会用 `pip` 管理依赖，了解虚拟环境。

4. **熟悉开发工具**
   学会使用 VS Code 或 PyCharm、命令行、Git；练习调试、阅读报错和编写基本测试。代码风格可参考 PEP 8。

5. **做综合练习**
   从命令行记账本、批量重命名、CSV 数据处理、爬取公开网页等小项目开始，再尝试把程序拆成模块、补充测试和文档。

6. **选择应用方向**
   根据目标继续深入：
   - **Web 后端**：HTTP、SQL、Flask/FastAPI 或 Django
   - **数据分析**：NumPy、pandas、Matplotlib、SQL
   - **机器学习/AI**：数学基础、NumPy、scikit-learn，再到 PyTorch
   - **自动化与运维**：文件处理、HTTP API、脚本、日志和部署
   - **测试开发**：pytest、接口测试、自动化测试

建议节奏：每天学习一点语法，并至少写一段代码；每学完一个阶段做一个能运行的小项目。初学者可以先从基础语法和 Python 3
环境开始。


 ========== 第3轮对话 ========== 

如果你的目标是**Java 后端开发**，可以按这条路线循序渐进：

### 1. 编程基础与 Java 入门
先安装 **JDK 17 或 21**，熟悉 IntelliJ IDEA，学习：

- 变量、数据类型、运算符、条件判断和循环
- 数组、方法、字符串
- 面向对象：类、对象、封装、继承、多态
- 接口、抽象类、异常处理
- 常用类：`String`、日期时间、包装类等

练习：学生成绩管理、简单计算器、猜数字游戏。

### 2. Java 核心知识
打牢日常开发常用的部分：

- 集合框架：`List`、`Set`、`Map`
- 泛型、枚举、注解
- 输入输出、文件操作
- Lambda、Stream API
- 多线程与并发基础
- JVM 基础：内存区域、垃圾回收、常见问题排查

同时学习 **Git**，掌握基本命令和代码版本管理。

### 3. 数据库与 SQL
先学 **MySQL**：

- 增删改查、排序、分组、连接查询
- 索引、事务、约束
- 基本的表设计与 SQL 优化思路

之后学习 JDBC，理解 Java 程序如何连接数据库。

### 4. Web 开发基础
了解：

- HTTP、请求与响应、JSON
- HTML、CSS、JavaScript 的基础概念
- Servlet、过滤器、监听器等基础知识

不用一开始深挖前端，但要理解浏览器、后端和数据库之间如何交互。

### 5. Java 后端主流框架
建议按这个顺序：

1. **Maven**：依赖管理与项目构建  
2. **Spring**：IoC、依赖注入、AOP  
3. **Spring Boot**：快速搭建后端应用  
4. **MyBatis 或 MyBatis-Plus**：数据库访问  
5. **Spring MVC**：接口开发  
6. **Spring Security 或 Sa-Token**：认证与权限管理  

之后再了解 Redis、消息队列、微服务等，不必一开始就追求“大而全”。

### 6. 做项目，形成完整开发能力
可以从一个简单的管理系统开始，逐步加入：

- 用户登录与权限
- 数据库设计和分页查询
- 参数校验与统一异常处理
- 日志、接口文档、单元

小谷哥哥: 如果你的目标是**Java 后端开发**，可以按这条路线循序渐进：

### 1. 编程基础与 Java 入门
先安装 **JDK 17 或 21**，熟悉 IntelliJ IDEA，学习：

- 变量、数据类型、运算符、条件判断和循环
- 数组、方法、字符串
- 面向对象：类、对象、封装、继承、多态
- 接口、抽象类、异常处理
- 常用类：`String`、日期时间、包装类等

练习：学生成绩管理、简单计算器、猜数字游戏。

### 2. Java 核心知识
打牢日常开发常用的部分：

- 集合框架：`List`、`Set`、`Map`
- 泛型、枚举、注解
- 输入输出、文件操作
- Lambda、Stream API
- 多线程与并发基础
- JVM 基础：内存区域、垃圾回收、常见问题排查

同时学习 **Git**，掌握基本命令和代码版本管理。

### 3. 数据库与 SQL
先学 **MySQL**：

- 增删改查、排序、分组、连接查询
- 索引、事务、约束
- 基本的表设计与 SQL 优化思路

之后学习 JDBC，理解 Java 程序如何连接数据库。

### 4. Web 开发基础
了解：

- HTTP、请求与响应、JSON
- HTML、CSS、JavaScript 的基础概念
- Servlet、过滤器、监听器等基础知识

不用一开始深挖前端，但要理解浏览器、后端和数据库之间如何交互。

### 5. Java 后端主流框架
建议按这个顺序：

1. **Maven**：依赖管理与项目构建  
2. **Spring**：IoC、依赖注入、AOP  
3. **Spring Boot**：快速搭建后端应用  
4. **MyBatis 或 MyBatis-Plus**：数据库访问  
5. **Spring MVC**：接口开发  
6. **Spring Security 或 Sa-Token**：认证与权限管理  

之后再了解 Redis、消息队列、微服务等，不必一开始就追求“大而全”。

### 6. 做项目，形成完整开发能力
可以从一个简单的管理系统开始，逐步加入：

- 用户登录与权限
- 数据库设计和分页查询
- 参数校验与统一异常处理
- 日志、接口文档、单元测试
- Redis 缓存
- Docker 部署

项目不求复杂，但要能讲清楚架构、关键代码和遇到的问题。

### 7. 进阶与求职准备
根据目标补充：

- 数据结构与算法基础
- 计算机网络、操作系统基础
- Linux 常用命令
- Redis、消息队列、分布式基础
- 项目复盘与面试常见问题

**推荐学习顺序总结：**

> Java 基础 → 集合与并发 → MySQL → Maven/Git → Spring Boot 与 MyBatis → 项目实战 → Redis、部署与进阶

刚开始不必同时学很多框架。把基础写扎实，每学一块就做练习，会比只看视频更有效。


 ========== 第4轮对话 ========== 

不一定要从零开始。建议先用几天做一次**基础回顾和自测**：很多内容可能只是暂时想不起来，重新看一遍会恢复得很快。

可以按这个顺序检查自己是否还熟悉：

1. **基础语法**：变量、条件判断、循环、数组、方法  
2. **面向对象**：类、对象、封装、继承、多态、接口  
3. **常用核心知识**：异常、集合、泛型、文件读写  
4. **小练习**：独立写一个命令行版学生管理系统，支持增删改查  

如果看完例子后能自己写出大部分代码，就从集合、泛型、异常等薄弱处补起；如果连基础语法和类对象都很难独立写出来，就从 Java 基础课程重新学一遍，但可以加快进度，不必把每节课都当第一次学。

重点是**边复习边写代码**。可以先安排 1 到 2 周回顾基础，再根据目标进入 Java 后端路线。
 ========== -> 第4轮对话结束 <- ========== 



小谷哥哥: 
不一定要从零开始。建议先用几天做一次**基础回顾和自测**：很多内容可能只是暂时想不起来，重新看一遍会恢复得很快。

可以按这个顺序检查自己是否还熟悉：

1. **基础语法**：变量、条件判断、循环、数组、方法  
2. **面向对象**：类、对象、封装、继承、多态、接口  
3. **常用核心知识**：异常、集合、泛型、文件读写  
4. **小练习**：独立写一个命令行版学生管理系统，支持增删改查  

如果看完例子后能自己写出大部分代码，就从集合、泛型、异常等薄弱处补起；如果连基础语法和类对象都很难独立写出来，就从 
Java 基础课程重新学一遍，但可以加快进度，不必把每节课都当第一次学。

重点是**边复习边写代码**。可以先安排 1 到 2 周回顾基础，再根据目标进入 Java 后端路线。


 ========== 第5轮对话 ========== 



### 1.2.1 keep_recent_messages() 的三个局限

上面的机器人能正常对话，但 1.1 中自己写的 `keep_recent_messages()` 有三个局限：

1. **截断后可能以 AI 消息开头**：机器人是先追加用户消息、再截断的。此时对话消息是奇数条，保留最后 2N 条，开头就会是一条 AI 回答，而它对应的问题已经被丢掉了；
2. **按轮数计算，控制不了 token 数**：一轮对话可能只有几个字，也可能有几千字。"保留 10 轮"不等于"不超过上下文窗口"；
3. **遇到工具调用会出错**：有工具调用时，一轮对话不再是 2 条消息（中间还有 AI 的工具调用消息和 ToolMessage）。按 2N 条截断，可能把 ToolMessage 和它前面的工具调用消息拆开，API 会直接报错。

用代码验证第 1、3 点（不调用模型，需要先运行 1.1 中定义函数的单元格）：

In [2]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# 局限 1：模拟机器人的第 4 轮 —— 先追加用户消息，再截断
chat = [SystemMessage("你是一个友好的助手")]
for i in range(1, 4):
    chat += [HumanMessage(f"问题{i}"), AIMessage(f"回答{i}")]
chat.append(HumanMessage("问题4"))
print("局限 1：", [f"{m.type}:{m.content}" for m in keep_recent_messages(chat, max_pairs=2)])

# 局限 3：包含两次工具调用的对话历史
tool_history = [
    SystemMessage("你是天气助手"),
    HumanMessage("北京天气怎么样？"),
    AIMessage("", tool_calls=[{"name": "get_weather", "args": {"city": "北京"}, "id": "call_1"}]),
    ToolMessage("北京：晴", tool_call_id="call_1"),
    AIMessage("北京今天是晴天。"),
    HumanMessage("上海呢？"),
    AIMessage("", tool_calls=[{"name": "get_weather", "args": {"city": "上海"}, "id": "call_2"}]),
    ToolMessage("上海：小雨", tool_call_id="call_2"),
    AIMessage("上海今天有小雨。"),
]
naive = keep_recent_messages(tool_history, max_pairs=3)
print("局限 3：", [m.type for m in naive])

局限 1： ['system:你是一个友好的助手', 'ai:回答2', 'human:问题3', 'ai:回答3', 'human:问题4']
局限 3： ['system', 'tool', 'ai', 'human', 'ai', 'tool', 'ai']


局限 1 的结果以 `ai:回答2` 开头；局限 3 的结果中，系统消息后面直接是一条 `tool` 消息，它前面的工具调用消息被截掉了。把局限 3 的结果发给模型：

In [3]:
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

try:
    model.invoke(naive + [HumanMessage("刚才查到的天气怎么样？一句话回答")])
except Exception as e:
    print(type(e).__name__, str(e)[:140])

OpenAIInvalidRequestError Error code: 400 - {'error': {'message': "Messages with role 'tool' must be a response to a preceding message with 'tool_calls' (request_id: 


报错的意思是：`tool` 角色的消息必须是对前面某条带 `tool_calls` 的消息的回应。

这些问题，LangChain 都提供了现成的函数来解决。

## 1.3 按 token 裁剪：trim_messages()

`trim_messages()` 是 LangChain 内置的裁剪函数（`01-messages的使用.ipynb` 的 1.5.2 节简单用过），能解决上面的三个问题。常用参数：

| 参数 | 作用 |
|---|---|
| `max_tokens` | 裁剪后最多保留的"token 数"（具体怎么计数由 `token_counter` 决定） |
| `token_counter` | 计数方式：传 `len` 表示**按消息条数**计数；传 `"approximate"` 表示按字符数**估算** token；也可以传一个函数或模型对象 |
| `strategy` | `"last"`（默认）保留最新的消息；`"first"` 保留最早的消息 |
| `include_system` | 为 `True` 时始终保留开头的系统消息，只在 `strategy="last"` 时使用 |
| `start_on` | 裁剪后的历史必须从哪种消息开始，一般设为 `"human"`。开头不是这种消息的，会继续往后丢，直到遇到为止 |
| `end_on` | 裁剪后的历史必须以哪种消息结束 |
| `allow_partial` | 放不下一整条消息时，是否保留这条消息的一部分，默认 `False` |

`start_on="human"` 是关键：它保证裁剪后的历史从用户消息开始，既不会以孤立的 AI 回答开头，也不会拆开工具调用。

先用 `token_counter=len`（按条数计数，效果类似 keep_recent_messages）修复上面的两个例子：

In [4]:
from langchain_core.messages import trim_messages

fixed1 = trim_messages(chat, max_tokens=5, token_counter=len, include_system=True, start_on="human")
print("局限 1 修复后：", [f"{m.type}:{m.content}" for m in fixed1])

fixed3 = trim_messages(tool_history, max_tokens=7, token_counter=len, include_system=True, start_on="human")
print("局限 3 修复后：", [m.type for m in fixed3])
print("发给模型不再报错：", model.invoke(fixed3 + [HumanMessage("刚才查到的天气怎么样？一句话回答")]).content[:60])

局限 1 修复后： ['system:你是一个友好的助手', 'human:问题3', 'ai:回答3', 'human:问题4']
局限 3 修复后： ['system', 'human', 'ai', 'tool', 'ai']


发给模型不再报错： 上海今天有小雨。


两个例子都从用户消息开始了：局限 1 丢掉了孤立的 `回答2`；局限 3 丢掉了孤立的 `tool` 和紧跟着的 AI 回答，只保留完整的"上海"那组对话。发给模型后也能正常回答，而且只提到了上海，因为模型只看得到这一组对话。

**按 token 计数时要注意中文**

`token_counter=len` 按条数计数，同样控制不了长度。真正要控制成本和上下文长度，要按 token 计数。但 `"approximate"`（即 `count_tokens_approximately()` 函数）默认按"约 4 个字符 = 1 个 token"估算，这是按英文设计的。下面和 DeepSeek 实际统计的输入 token 数对比一下（设置 `max_tokens=1`，只为读取输入 token 数）：

In [5]:
from functools import partial
from langchain_core.messages import HumanMessage
from langchain_core.messages.utils import count_tokens_approximately
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

counter_model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},
    max_tokens=1,  # 只生成 1 个 token，用来读取真实的输入 token 数
)

zh = [HumanMessage("请详细解释一下 Python 中列表、元组、字典和集合这四种数据结构的区别，以及它们各自适合在什么场景下使用。" * 3)]
en = [HumanMessage("Please explain in detail the differences between lists, tuples, dictionaries and sets in Python, and when each of them should be used. " * 3)]
zh_counter = partial(count_tokens_approximately, chars_per_token=1.8)  # 把“每个 token 约几个字符”调小

for label, msgs in [("中文", zh), ("英文", en)]:
    actual = counter_model.invoke(msgs).usage_metadata["input_tokens"]
    print(f"{label}：{len(msgs[0].content)} 个字符 | 默认估算 {count_tokens_approximately(msgs)} | "
          f"chars_per_token=1.8 估算 {zh_counter(msgs)} | 实际 {actual}")

中文：165 个字符 | 默认估算 46 | chars_per_token=1.8 估算 97 | 实际 97


英文：405 个字符 | 默认估算 106 | chars_per_token=1.8 估算 231 | 实际 83


结果说明：

- **中文**：默认参数估算 46，实际 97，**低估了一半多**；把 `chars_per_token` 调成 1.8 后，估算值与实际一致；
- **英文**：默认参数估算 106，实际 83，略微高估；而用 1.8 会严重高估（231）。

所以**中文对话要调小 `chars_per_token`（比如 1.8），英文用默认值即可**，中英混合的对话介于两者之间。估算宁可偏高（多裁掉一点），也不要偏低（可能超出上下文窗口）。

> 也可以把模型对象传给 `token_counter`，但 ChatDeepSeek 继承自 OpenAI 的实现，用的是 OpenAI 的分词器（tiktoken），对 DeepSeek 同样只是近似值。最准确的数字只有模型返回的 `usage_metadata`。

下面把 1.2 的机器人改成**按 token 裁剪**（用固定的问题代替 `input()`，方便直接运行），每轮调用前把历史裁剪到约 150 token 以内：

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage, trim_messages
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

questions = [
    "你好，我叫小明，是一名刚开始学 Python 的大学生",
    "列表和元组有什么区别？用两三句话说明",
    "字典适合用在什么场景？用两三句话说明",
    "你还记得我是谁吗？",
]
messages = [SystemMessage("你是一个耐心的 Python 导师")]
for question in questions:
    messages.append(HumanMessage(question))
    messages = trim_messages(  # 每轮调用前裁剪：保留系统消息，最多约 150 token，从用户消息开始
        messages, max_tokens=150, token_counter=zh_counter, include_system=True, start_on="human"
    )
    response = model.invoke(messages)
    messages.append(response)
    print(f"用户：{question}")
    print(f"  发送了 {len(messages) - 1} 条消息，估算 {zh_counter(messages[:-1])} token，"
          f"实际输入 {response.usage_metadata['input_tokens']} token")
    print(f"  AI：{response.content[:60]}\n")

用户：你好，我叫小明，是一名刚开始学 Python 的大学生
  发送了 2 条消息，估算 37 token，实际输入 22 token
  AI：你好，小明！很高兴认识你！🎉

欢迎来到 Python 的世界，作为初学者选择 Python 是非常棒的决定——它语法简



用户：列表和元组有什么区别？用两三句话说明
  发送了 2 条消息，估算 32 token，实际输入 21 token
  AI：列表（list）是可变的，创建后可以增删改元素，用方括号 `[]` 表示；元组（tuple）是不可变的，创建后不能修改，



用户：字典适合用在什么场景？用两三句话说明
  发送了 4 条消息，估算 127 token，实际输入 114 token
  AI：字典（dict）适合用**键值对**快速查找、插入和删除数据的场景，比如用学号查学生信息、用单词查释义。它的查找平均是 



用户：你还记得我是谁吗？
  发送了 4 条消息，估算 110 token，实际输入 99 token
  AI：我不记得你是谁。每次对话我都是从零开始，没有关于你身份、历史对话或之前交流的记忆。

如果你想让我了解你的背景（比如你在



观察结果：

- 每轮发送的内容都控制在 150 token 以内（估算值），实际输入的 token 数也没有随着轮数不断增长；
- 第 2 轮只发送了 2 条消息（系统消息 + 当前问题）：第 1 轮 AI 的欢迎语比较长，加上它会超过 150 token，所以 `trim_messages()` 把第 1 轮整组丢掉了。因为设置了 `start_on="human"`，不会只留下半组对话；
- 第 4 轮问"你还记得我是谁吗？"，模型已经不记得"小明"了，因为自我介绍在第 1 轮，早就被裁掉了。这再次说明：**裁剪只能控制长度，保不住早期的关键信息**，1.6 节的摘要可以解决这个问题。

实际应用中 `max_tokens` 要设得大得多（比如上下文窗口的一部分），这里设成 150，只是为了在几轮之内就能看到裁剪效果。

## 1.4 过滤消息：filter_messages()

有时发给模型前，需要去掉某些消息，比如群聊中只保留某几个人的发言、去掉工具调用的中间过程。`filter_messages()` 按类型、名称（`name`）或 ID 筛选消息：

| 参数 | 作用 |
|---|---|
| `include_types` / `exclude_types` | 保留 / 去掉某些类型的消息，可以写 `"human"` 等字符串，也可以写 `HumanMessage` 等类 |
| `include_names` / `exclude_names` | 按消息的 `name` 字段保留 / 去掉 |
| `include_ids` / `exclude_ids` | 按消息的 `id` 字段保留 / 去掉 |
| `exclude_tool_calls` | 为 `True` 时去掉所有带工具调用的 AIMessage 和所有 ToolMessage，也可以传入要去掉的工具调用 ID 列表 |

In [7]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, filter_messages

group_chat = [
    SystemMessage("你是群聊助手"),
    HumanMessage("大家好，我是小明", name="小明", id="1"),
    HumanMessage("我是小红，想讨论周末去哪玩", name="小红", id="2"),
    AIMessage("欢迎两位！可以说说各自的想法。", id="3"),
    HumanMessage("我想去爬山", name="小明", id="4"),
]
print("只保留用户消息：", [m.content for m in filter_messages(group_chat, include_types=["human"])])
print("去掉小红的消息：", [m.content for m in filter_messages(group_chat, exclude_names=["小红"])])
print("去掉 id 为 3 的消息：", [m.content for m in filter_messages(group_chat, exclude_ids=["3"])])
print("去掉工具调用过程：", [m.type for m in filter_messages(tool_history, exclude_tool_calls=True)])

只保留用户消息： ['大家好，我是小明', '我是小红，想讨论周末去哪玩', '我想去爬山']
去掉小红的消息： ['你是群聊助手', '大家好，我是小明', '欢迎两位！可以说说各自的想法。', '我想去爬山']
去掉 id 为 3 的消息： ['你是群聊助手', '大家好，我是小明', '我是小红，想讨论周末去哪玩', '我想去爬山']
去掉工具调用过程： ['system', 'human', 'ai', 'human', 'ai']


最后一行中，两组"工具调用消息 + ToolMessage"都被去掉了，只剩下用户的问题和 AI 的最终回答。对话很长时，这样可以省掉工具返回的大量中间数据。

## 1.5 合并连续的同类消息：merge_message_runs()

消息列表中可能出现连续多条同类消息，比如用户连发了两条消息、写了多条系统消息，或者过滤之后原本不相邻的用户消息挨在了一起。部分模型接口不允许连续出现同一角色的消息，或者只接受一条系统消息。`merge_message_runs()` 会把连续的同类消息合并成一条，内容之间默认用换行（`chunk_separator` 参数）连接：

In [8]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, merge_message_runs

messages = [
    SystemMessage("你是编程助手"),
    SystemMessage("回答要简短"),
    HumanMessage("我想学编程"),
    HumanMessage("推荐一门入门语言"),
    AIMessage("推荐 Python。"),
]
for m in merge_message_runs(messages):
    print(type(m).__name__, repr(m.content))

SystemMessage '你是编程助手\n回答要简短'
HumanMessage '我想学编程\n推荐一门入门语言'
AIMessage '推荐 Python。'


## 1.6 用摘要代替丢弃

1.1 的例子中，裁剪导致模型忘记了第一个问题。另一种做法是：**把较早的对话交给模型总结成一段摘要**，用摘要代替原来的消息，再保留最近几轮的原文。这样既控制了长度，又保留了关键信息。

步骤：

1. 把要压缩的旧消息用 `get_buffer_string()` 转成"Human: …\nAI: …"形式的对话文本；
2. 让模型生成摘要。**摘要一定会丢失细节**，所以提示词中要写清楚必须保留的内容（这里要求逐条保留用户的问题）；
3. 新的历史 = 系统消息 + 摘要 + 最近几轮原文。

下面用 1.1 中同样的三轮对话演示（为了不重新调用模型，这里直接写出了当时的问答）：

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, get_buffer_string
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式，回答更快
)

history = [
    SystemMessage("你是一个专业的python导师"),
    HumanMessage("什么是列表？用一句解释"),
    AIMessage("列表是 Python 中有序、可变、可以存放任意类型元素的序列，用方括号 [] 表示。"),
    HumanMessage("列表和元组有什么区别？用一句解释"),
    AIMessage("列表可变、用 [] 表示，元组不可变、用 () 表示。"),
    HumanMessage("什么是字典呢？用一句解释"),
    AIMessage("字典是由键值对组成的可变映射类型，用 {} 表示，通过键快速查找值。"),
]
old_messages, recent_messages = history[1:-2], history[-2:]  # 前两轮压缩成摘要，最后一轮保留原文

print("1. 要压缩的对话：\n" + get_buffer_string(old_messages) + "\n")
summary = model.invoke([
    SystemMessage("请把下面的对话压缩成摘要：按顺序逐条列出用户问过的每个问题（保留原文）和结论，不超过 150 字。"),
    HumanMessage(get_buffer_string(old_messages)),
]).content
print("2. 摘要：\n" + summary + "\n")

new_history = [history[0], SystemMessage(f"以下是之前对话的摘要：\n{summary}")] + recent_messages
new_history.append(HumanMessage("我第一个问题问的是什么？"))
print("3. 回答：", model.invoke(new_history).content)

1. 要压缩的对话：
Human: 什么是列表？用一句解释
AI: 列表是 Python 中有序、可变、可以存放任意类型元素的序列，用方括号 [] 表示。
Human: 列表和元组有什么区别？用一句解释
AI: 列表可变、用 [] 表示，元组不可变、用 () 表示。



2. 摘要：
用户问题：1.什么是列表？2.列表和元组有什么区别？结论：1.列表是Python有序可变、可存任意类型元素的序列，用[]表示。2.列表可变、用[]，元组不可变、用()。



3. 回答： 你第一个问题是：**什么是列表？**


这次模型答对了第一个问题，和 1.1 形成对比。

摘要的代价是**每次压缩都要额外调用一次模型**，有成本和延迟。所以实际应用中，一般在历史超过一定长度时才触发压缩，而不是每轮都做。

LangChain 1.x 的 Agent 提供了现成的摘要中间件 `SummarizationMiddleware`，它会在对话 token 数达到阈值时自动压缩（阈值可以按模型上下文窗口的比例设置，用到的就是第 2 章 `07-profile-initparams-config.ipynb` 中模型画像的 `max_input_tokens`），后面学 Agent 时会用到。

> 对话历史还需要**保存**：程序重启后，内存中的消息列表就没了。保存方法见 `01-messages的使用.ipynb` 1.5.3 节的 `messages_to_dict()`；后面学 LangGraph 时，会用它的检查点（checkpointer）机制自动保存。

## 1.7 拓展-消息属性：content、content_blocks

### 1.7.1 content
消息的 content 可以理解为数据内容，它是弱类型的，支持字符串和列表（列表元素通常为字典）。

举例1：存储字符串

如果只是纯文本内容，直接传递字符串就好。

In [12]:
from langchain_core.messages import HumanMessage

msg1 = HumanMessage(content = "你好啊")
msg2 = HumanMessage("你好啊")
print(msg1)
print(msg2)

content='你好啊' additional_kwargs={} response_metadata={}
content='你好啊' additional_kwargs={} response_metadata={}


举例2: 存储字典列表

如果需要发送的不只是文本，如多模态内容，则需要content的 字典列表 形式。

字典内容遵循模型供应商的API规范，以 openai: gpt-4.1 为例。

参考官方文档：https://developers.openai.com/api/reference/python/resources/chat/subresources/completions/methods/create

In [13]:
# Provider-native format (e.g., OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])
print(human_message)


content=[{'type': 'text', 'text': 'Hello, how are you?'}, {'type': 'image_url', 'image_url': {'url': 'https://example.com/image.jpg'}}] additional_kwargs={} response_metadata={}


In [23]:
import base64


def encode_image(img_path, img_type='jpeg'):
    """
    将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据
    """
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode('utf-8')}"

# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

# 图像路径
img_path = "/Users/skk/Developer/lang-chain-learn/image/image_test.png"
base64_img = encode_image(img_path)
#print("Base64 编码后的图片字符串:", base64_img)
response = model.invoke([
    HumanMessage(content=[
        {"type": "text", "text": "这张图有什么?"},
        {"type": "image_url", "image_url": {"url": base64_img}}
    ])
])
print("模型回答:", response.content)


模型回答: 图中是一个放在浅色台面上的金色香水瓶，瓶身细长，顶部有透明瓶盖。背景是米白色的墙面。


### 1.7.2 content_blocks

在 LangChain 1.x 中， content_blocks 是消息对象（BaseMessage）的一项重大升级。它的核心目标是提供一种跨模型供应商、标准化的多模态数据结构。

过去，处理图片、音频、甚至是模型生成的“思维链（Reasoning）”内容时，不同供应商（OpenAI,Anthropic, Google 等）的 API 格式各异，导致开发者需要写大量的适配代码。 content_blocks 的出现终结了这种混乱。


在 LangChain 1.2 版本中，消息对象的 content 属性依然存在（为了向前兼容），但新增了content_blocks 属性，可以将 content 解析为标准、类型安全的表示。

* 数据结构：它是一个 list[TypedDict] 。
* 统一格式：每个 block 都有一个 type 字段，用于区分内容类型。
* 支持类型：包括 text （文本）、 image （图片）、 audio （音频）、 video （视频）、tool_call （工具调用）以及 reasoning （推理/思维链）。

支持的字段类型：https://docs.langchain.com/oss/python/langchain/messages#openai

也就是说content 和 content_blocks 角色是一致的，但content目前还存在只是为了和旧版模型保持兼容。

1. 输入格式化

对于复杂的对话（带图片或工具结果），建议使用 content_blocks 列表形式构建 HumanMessage或 AIMessage 。

借助 content_blocks ，我们可以用一套标准代码，无缝地在不同厂商的模型之间切换。

举例1: OpenAI 模型

In [ ]:
img_path = "/Users/skk/Developer/lang-chain-learn/image/image_test.png"
base64_img = encode_image(img_path)
#print("Base64 编码后的图片字符串:", base64_img)
# 从.env文件中加载环境变量
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_API_BASE = os.environ["OPENAI_API_BASE"]  # 与本项目 .env 中的名称一致

model = ChatOpenAI(
    model="gpt-6-luna",  # 当前配置已验证可用；原 gpt-5.5 返回模型权限 403
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_API_BASE,
)

response = model.invoke([
    HumanMessage(content_blocks=[
        {"type": "text", "text": "这张图有什么?"},
        {"type": "image", "url": base64_img}
    ])
])
print("模型回答:", response.content)

模型回答: 图中是一支金色的按压式护肤品瓶，立在米色背景上。瓶身上的文字看不清。


举例2: 看Deepseek

In [31]:
def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据
    """
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")



img_path = "/Users/skk/Developer/lang-chain-learn/image/image_test.png"
base64_img = encode_image(img_path)

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    # provider:model_name 提供商:模型名称
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    temperature=0.5, # 温度参数(optional) 表示模型的随机性，0表示确定性，1表示随机性最高
    # **kwargs 其他参数，根据模型不同而不同
    extra_body={"thinking": {"type": "enabled"}},
    base_url=DEEPSEEK_BASE_URL,
)
#print("Base64 编码后的图片字符串:", base64_img)
response = model.invoke([
    HumanMessage(content_blocks=[
        {"type": "text", "text": "这张图有什么?"},
        {
            "type": "image",
            "base64": base64_img,
            "mime_type": "image/png",
        },
    ])
])
print("模型回答:", response.content)

模型回答: 这张图片展示的是一瓶 **雅诗兰黛（Estée Lauder）的粉底液**。

具体细节如下：
*   **产品外观**：它是一个长方体的透明玻璃瓶，配有金色的按压泵头和透明的方形外盖。
*   **标识**：瓶身正面有金色的品牌标志，上方写着“ESTÉE LAUDER”，下方有小字标注产品信息（从颜色和瓶身比例来看，通常是该品牌的“Double Wear”持妆粉底液）。
*   **内容物**：瓶内装有米色/浅肤色的液体粉底。
*   **背景与光影**：背景是温暖的米黄色调，光线从左侧打来，在瓶身右侧投射出清晰的阴影，整体呈现出简洁、高级的产品摄影风格。


2. 输出格式化

content_blocks 还可用于输出格式化，以deepseek官网的 deepseek-v4-flash 为例，其输出包含思考内容，后者位于 additional_kwargs 的 reasoning_content 字段下。比如：

In [20]:

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    # provider:model_name 提供商:模型名称
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    temperature=0.5, # 温度参数(optional) 表示模型的随机性，0表示确定性，1表示随机性最高
    # **kwargs 其他参数，根据模型不同而不同
    extra_body={"thinking": {"type": "enabled"}},
    base_url=DEEPSEEK_BASE_URL,
)

# 调用模型
response = model.invoke("用一句话介绍 LangSmith")
rich_print("模型回答:", response)


模型回答:
AIMessage(
    content='LangSmith 是 LangChain 推出的、用于大语言模型应用调试、测试、评估与监控的一体化开发平台。',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '我们需要回答用户中文请求：“用一句话介绍 
LangSmith”。需要一句话介绍。需要准确。LangSmith 是 LangChain 推出的平台？用于 LLM 
应用开发、调试、测试、评估和监控。一句话。可能需要中文。应简洁。可以：“LangSmith 是 LangChain 
推出的用于大语言模型应用的调试、测试、评估与监控的一体化开发平台。” 这是一句话。确保不违反? 
用户要求一句话介绍。直接给出。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 125,
            'prompt_tokens': 35,
            'total_tokens': 160,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 97,
                'rejected_prediction_tokens': None,
                'text_tokens': None
            },
            'prompt_tokens_details': {
                'audio_tokens': None,
                'cache_write_tokens': None,
                'cached_tokens': 0,
                'image_tokens': None,
                'text_tokens': None
            },
            'prompt_cache_hit_tokens': 0,
            'prompt_cache_miss_tokens': 35
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-flash',
        'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669',
        'id': '8ad210dd-61b3-4c30-8f38-8bc9bc86f182',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a0d494-a55e-7023-bf98-06aba42453da-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 35,
        'output_tokens': 125,
        'total_tokens': 160,
        'input_token_details': {'cache_read': 0},
        'output_token_details': {'reasoning': 97}
    }
)

不同的模型其输出格式可能不同，仅为提取思考内容，切换模型都可能需要更改代码，非常不方便。

content_blocks提供了 统一的输出格式 ，可以将不同格式的响应统一为标准格式。

注意：content_blocks是 懒加载 的，即调用时才会解析。

In [32]:

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

model = init_chat_model(
    # provider:model_name 提供商:模型名称
    model="deepseek:deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    # **kwargs 其他参数，根据模型不同而不同
    extra_body={"thinking": {"type": "enabled"}},
    base_url=DEEPSEEK_BASE_URL,
)

response = model.invoke("你好，一句话回答")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)

==================== -> response <- ====================
content='你好，请告诉我你想问什么，我会用一句话回答。' additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答用户。用户说“你好，一句话回答”。这可能是要求我们只回答一句话？但没具体问题。我们需要用一句话回应。可能应该问候并询问需要什么帮助，且保持一句话。用中文，一句话。可以：“你好，请告诉我你想问什么，我会用一句话回答。” 这本身一句话。需要确保没有多余。最终只输出这一句。'} response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 34, 'total_tokens': 127, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 79, 'rejected_prediction_tokens': None, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 34}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '0d5468b3-545a-4df7-a579-2f256b73a444', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0d4a2-2202

### 1.7.3 补充：content 与 content_blocks 的关系

1.7.2 中"content 目前还存在只是为了和旧版模型保持兼容"的说法不太准确（另外，content_blocks 从 LangChain 1.0 起就有了）。更准确的理解是：

- **`content` 是消息真正存储的数据**，发给模型的、序列化保存的都是它；
- **`content_blocks` 是一个属性**，每次访问时把 `content`（以及 `additional_kwargs` 中的思考过程等）按统一格式解析出来，本身不存储数据，这就是"懒加载"的含义；
- 用 `content_blocks=` 构造消息时，LangChain 会把这些统一格式的内容块**存进 `content`**；
- 初始化模型时设置 `output_version="v1"`（第 2 章 07 笔记 8.4 节），模型返回的 `content` 本身就是统一格式的内容块列表。

下面验证这几点：

In [10]:
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os
from langchain.chat_models import init_chat_model
load_dotenv(override=True, verbose=True,dotenv_path="/Users/skk/Developer/lang-chain-learn/conf/.env")

DEEPSEEK_API_KEY = os.environ["DEEPSEEK_API_KEY"]
DEEPSEEK_BASE_URL = os.environ["DEEPSEEK_BASE_URL"]

# 1. 用 content_blocks 构造消息，数据实际存进了 content
msg = HumanMessage(content_blocks=[
    {"type": "text", "text": "这张图有什么?"},
    {"type": "image", "url": "https://example.com/image.png"},
])
print("1. content：", msg.content, "\n")

# 2. 默认情况：content 是字符串，思考过程在 additional_kwargs 中，content_blocks 把两者统一解析出来
model = init_chat_model(model="deepseek:deepseek-v4-flash", api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL)
response = model.invoke("你好，一句话回答")
print("2. content：", repr(response.content))
print("   思考过程在 additional_kwargs 中：", "reasoning_content" in response.additional_kwargs)
print("   content_blocks 的类型：", [block["type"] for block in response.content_blocks], "\n")

# 3. output_version="v1"：content 本身就是统一格式的内容块列表
v1_model = init_chat_model(
    model="deepseek:deepseek-v4-flash", api_key=DEEPSEEK_API_KEY, base_url=DEEPSEEK_BASE_URL, output_version="v1"
)
v1_response = v1_model.invoke("你好，一句话回答")
print("3. content 的类型：", type(v1_response.content).__name__, [block["type"] for block in v1_response.content])
print("   用 .text 取出回答文本：", v1_response.text)

1. content： [{'type': 'text', 'text': '这张图有什么?'}, {'type': 'image', 'url': 'https://example.com/image.png'}] 



2. content： '你好，有什么我可以帮你的吗？'
   思考过程在 additional_kwargs 中： True
   content_blocks 的类型： ['reasoning', 'text'] 



3. content 的类型： list ['reasoning', 'text']
   用 .text 取出回答文本： 你好，请问有什么可以帮你？


## 1.8 总结

**管理对话历史的几种方法**

| 方法 | 做法 | 优点 | 缺点 / 注意 |
|---|---|---|---|
| 保存完整历史 | 每轮追加 HumanMessage 和 AIMessage | 最简单，信息不丢失 | 越来越长，成本高，最终会超过上下文窗口 |
| 只保留最近 N 轮（1.1） | 自己写函数截取最后 2N 条 | 简单直观 | 控制不了 token 数；可能以 AI 消息开头；会拆开工具调用 |
| 按 token 裁剪（1.3） | `trim_messages()` | 能精确控制长度；`start_on="human"` 保证结构完整 | 早期信息被遗忘；中文要调小 `chars_per_token` |
| 过滤（1.4） | `filter_messages()` | 按类型、名称、ID 去掉不需要的消息 | 只是筛选，不压缩内容 |
| 合并（1.5） | `merge_message_runs()` | 合并连续的同类消息，满足部分接口的格式要求 | — |
| 摘要（1.6） | 模型总结旧消息 + 保留最近几轮 | 控制长度的同时保留关键信息 | 需要额外调用模型；摘要会丢失细节 |

**要点回顾**

1. 大模型没有记忆，每次调用都要传完整（或处理过）的对话历史，并把模型的回答追加回去。
2. 裁剪历史时要保证结构完整：从用户消息开始，不能拆开工具调用消息和 ToolMessage，否则 API 会报错。
3. `trim_messages()` 的 `token_counter`：`len` 按条数，`"approximate"` 按字符数估算（中文会低估），模型返回的 `usage_metadata` 才是准确值。
4. 裁剪、过滤、合并可以组合使用，比如先 `filter_messages()` 去掉工具调用过程，再 `merge_message_runs()`，最后 `trim_messages()`。
5. 需要长期记住关键信息时，用摘要代替丢弃；Agent 中可以直接使用 `SummarizationMiddleware`。
6. `content` 是消息实际存储的数据，`content_blocks` 是按统一格式读取它的视图。